# ⚠️ 2025 winning frequency model — historical, AutoCarver **7.0.5**

**This notebook is a period artifact. Do not copy its AutoCarver API into a new
project.** It is the run that won [ENS *Challenge Data*
#161](https://challengedata.ens.fr/challenges/161) in 2025, kept exactly as it
was executed so the "then vs now" comparison in `RESULTS.md` is reproducible.

The dangerous part is that it still **imports and runs** on a current AutoCarver
and silently does something else:

| 2025 here | On the pinned version |
|---|---|
| `MulticlassCarver` — carves **one-vs-rest**, several columns per feature | `MulticlassCarver` means **one** carving per feature against the full crosstab. The 2025 behaviour is now `OneVsRestCarver` |
| single-process carving | `ProcessingConfig(n_jobs=...)` carves across a process pool |
| selectors take positional per-type budgets | `n_best_features=` keyword + `SelectionConfig` |

For the current pipeline see **`frequency_model_2026.ipynb`**.


# Loading Data & Processing

First, let's load the challenge's data and target and join them on ``ID``

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv")
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv")
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

<tmp>/ipykernel_2500\2176795849.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path+'train_input_Z61KlZo.csv')


x_train (383610, 373)
y_train (383610, 4)
data (383610, 376)


## Defining Target

Has stated in the data dictionary, the number of claims is ``FREQ`` x ``ANNEE_ASSURANCE``, it will be our target variable

In [2]:
target_col = "TARGET"
data[target_col] = (data["FREQ"] * data["ANNEE_ASSURANCE"]).astype(int)
data[target_col].value_counts(normalize=False).sort_index()

TARGET
0    381061
1      2458
2        87
3         2
4         1
5         1
Name: count, dtype: int64

## Stratified Sampling & Weighting

**Stratified Sampling** is applied to ensure same distribution between train (80%) and dev (20%) samples

**Weights** are computed as the inverse frequency of each class of the target, giving more weights to rare classes like (ie observed claims)




**Note:** observations with more than 2 claims are scarce (only 4 of them), they are merged with those that have 2 claims, defining the following multiclass target `Y`:
* `0`: no claim
* `1`: one claim
* `2`: two or more claims

In [3]:
import numpy as np

from collections import Counter
from sklearn.model_selection import train_test_split

y_transform = lambda u: u.where(
    u <= 1, 2
)  # Transform the target variable to ensure binary classification

# Compute class frequencies
class_counts = Counter(y_transform(data[target_col]))
total_samples = len(data[target_col])

# Compute inverse frequency class weights
class_weights = {
    cls: total_samples / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
print("Class weights:", class_weights)

# Assign sample weights based on target values
weights = np.array([class_weights[label] for label in y_transform(data[target_col])])

# Train-test split
x_train, x_dev, y_train, y_dev, w_train, w_dev = train_test_split(
    data,
    data[target_col],
    weights,
    test_size=0.2,
    random_state=42,
    stratify=y_transform(data[target_col]),
)
print("y_train mean", y_train.mean())
print("y_dev mean", y_dev.mean())

Class weights: {0: 0.3355630725789309, 1: 52.0219690805533, 2: 1405.1648351648353}
y_train mean 0.006895023591668622
y_dev mean 0.006921091733792133


In [4]:
y_train.value_counts(normalize=True)

TARGET
0    0.993356
1    0.006406
2    0.000231
5    0.000003
3    0.000003
Name: proportion, dtype: float64

In [5]:
y_dev.value_counts(normalize=True)

TARGET
0    0.993353
1    0.006413
2    0.000209
3    0.000013
4    0.000013
Name: proportion, dtype: float64

# Feature Engineering

The `Processor` class is used to build several new features:
* `ZONE` is converted to `ZONE_REGION`
* Copies of `ALTITUDE_xxx`, `IND_xxx`, `MEN_xxx` and `LOG_xxx` are converted to numerical features: `ALTITUDE_xxx_num`, `IND_xxx_num`, `MEN_xxx_num` and `LOG_xxx_num`
* Vetusty of buildings is computed: `LOG_VETUSTE`
* It learns the distribution of `LOG_TOT`, `LOG_VETUSTE`, `MEN_TOT`, `IND_TOT`, `IND_SNV`,`ALTITUDE_TOT` per `ZONE_REGION`on train sample. It's then used to compute the ratio of each sample to its mean per region (sort of a measure of divergence from the regional mean)
* Per observation, `CA_TOT` and `CA_MEAN` are computed as the sum and mean of `CA1`, `CA2` and `CA3`. `CA_TOT` and `CA_MEAN` are used to compute ratios with  `CA1`, `CA2` and `CA3`
* Open Source data from [Base de Données sur les Incendies de Forêts en France](https://bdiff.agriculture.gouv.fr/) are added: fire extinction rates per `ZONE` in 2023, total surfaces burnt in 2023 and in the 2016-2020 period per `ZONE`, surface burnt over surface of forest per `ZONE` in 2023. Those features are crossed with `NB_CASERNES` and `ZONE_VENT`
* Numerical `KAPITAL_xxx` columns are sumed and maxed out into `KAPITAL_SUM` and `KAPITAL_MAX`
* Temperature columns are crossed with one another
* Binary non-numerical columns are one hot encoded

In [ ]:
import warnings
from data_toolkit import Processor

warnings.simplefilter(action="ignore", category=FutureWarning)

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)

# Feature Processing

## Sorting features per data type

* numerical features
* categorical features
* ordinal features (with their respective ordering)

In [ ]:
import numpy as np

# get the columns that are categorical
categorical_columns = x_train.select_dtypes(include=["object"]).columns

# get the columns that are numerical
numerical_columns = x_train.select_dtypes(include=["int64", "float64"]).columns

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
# ordinals += ["AN_EXERC"]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# get the columns that are to be removed
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += [
    "DEROG13",
    "DEROG14",
    "DEROG16",
]  # no values
# columns that were not available at the time of the frequency model
to_remove += [
    "DEROG13_formatted",
    "DEROG8_formatted",
    "DEROG3_formatted",
    "DEROG16_formatted",
    "DEROG14_formatted",
    "KAPITAL_MAX",
    "KAPITAL_SUM",
]
to_remove += ["IND_Y1_Y2_num", "IND_INC_num"]

# removing columns
categorical_columns = [
    col
    for col in categorical_columns
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col
    for col in numerical_columns
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

91 219 238 548


In [39]:
for c in categorical_columns:
    print(f"'{c}': {np.sort(np.unique([v for v in data[c].value_counts().index]))}")

'ACTIVIT2': ['ACT1' 'ACT2' 'ACT3' 'ACT4' 'ACT5' 'ACT6' 'ACT7' 'ACT8' 'ACT9']
'VOCATION': ['VOC1' 'VOC2' 'VOC3' 'VOC4' 'VOC5' 'VOC6' 'VOC7' 'VOC8']
'ADOSS': ['N' 'O']
'CARACT1': ['N' 'O' 'R']
'CARACT3': ['N' 'O' 'R']
'INDEM1': ['N' 'O']
'TYPBAT1': ['gibier-plumes' 'lapin' 'porcs' 'veaux' 'volaille']
'INDEM2': ['CLASS1' 'CLASS10' 'CLASS11' 'CLASS13' 'CLASS2' 'CLASS3' 'CLASS4'
 'CLASS5' 'CLASS6' 'CLASS7' 'CLASS8' 'CLASS9']
'FRCH1': ['0' '1' '2' '3' 'i']
'FRCH2': ['1' '2' '3' '4' '5' 'A']
'DEROG2': ['N' 'O']
'DEROG3': ['N' 'O']
'DEROG4': ['N' 'O']
'DEROG5': ['N' 'O']
'DEROG8': ['N' 'O']
'DEROG12': ['D03' 'D04' 'D12' 'D18' 'D35']
'KAPITAL34': ['N' 'O']
'KAPITAL35': ['N' 'O']
'KAPITAL37': ['N' 'O']
'KAPITAL40': ['N' 'O']
'KAPITAL41': ['N' 'O']
'KAPITAL42': ['N' 'O']
'KAPITAL43': ['N' 'O']
'RISK6': ['A' 'N' 'O']
'RISK8': ['N' 'O']
'RISK9': ['N' 'O' 'R']
'RISK10': ['N' 'O' 'R']
'RISK11': ['N' 'O' 'R']
'RISK12': ['N' 'O' 'R']
'RISK13': ['N' 'O' 'R']
'EQUIPEMENT2': ['N' 'O' 'R']
'EQUIPEMENT5': [

## Processing Qualitative Features

For a binary target variable, following processing is applied:
* Pre-processing of ordinals:
    - ordering modalities according to user-provided values
    - grouping modalities with less than `min_freq=2%` frequency into their closest modality (previous or next modality) according to target rate (train sample)
* Pre-processing of categoricals:
    - grouping modalities with less than `min_freq=2%` frequency into a dedicated one (train sample)
    - ordering modalities according to target rate (train sample)
* All combinations of up to `max_n_mod=5` modalities are sorted by Tschuprow's T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/2=1%` frequency)
    - distinct target rate per consecutive modalities
    - no inversion of target rates between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [ ]:
from AutoCarver import Features, MulticlassCarver

# defining the features to carve
features = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

# defining the carver
carver = MulticlassCarver(
    features=features,
    min_freq=0.02,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)

# carving train data and testing robustness on dev data
x_train = carver.fit_transform(
    x_train, y_transform(y_train), X_dev=x_dev, y_dev=y_transform(y_dev)
)

In [ ]:
# version = "010"
# carver.save(f"model/frequency/{version}_carver.json", light_mode=True)

In [9]:
from AutoCarver import MulticlassCarver

version = "010"
carver = MulticlassCarver.load(f"model/frequency/{version}_carver.json")
carver.summary

content  \
feature                          label                                                      
Categorical('ACTIVIT2__y=1')     0                                     [ACT3, ACT8, ACT2]   
                                 1                                           [ACT9, ACT1]   
                                 2                    [ACT4, ACT6, ACT7, __OTHER__, ACT5]   
Categorical('VOCATION__y=1')     0        [VOC7, VOC5, VOC3, VOC2, __OTHER__, VOC8, VOC1]   
                                 1                                           [VOC6, VOC4]   
...                                                                                   ...   
Ordinal('LOG_SOC__y=2')          1      [04. <= 40, 05. <= 50, 06. <= 60, 07. <= 70, 0...   
Ordinal('DISTANCE_142__y=2')     0                                  [02. <= 17, 01. <= 8]   
                                 1                               [04. >= 946, 03. <= 946]   
Ordinal('FXI3SAB_VOR_MM_A__y=2') 0                                               01. <= 8   
                                 1                      [04. >= 24, 03. <= 24, 02. <= 17]   

                                        target_mean  frequency  
feature                          label                          
Categorical('ACTIVIT2__y=1')     0         0.000000   0.066037  
                                 1         0.000220   0.785023  
                                 2         0.000438   0.148940  
Categorical('VOCATION__y=1')     0         0.000010   0.322512  
                                 1         0.000346   0.677488  
...                                             ...        ...  
Ordinal('LOG_SOC__y=2')          1         0.000840   0.023269  
Ordinal('DISTANCE_142__y=2')     0         0.000303   0.375824  
                                 1         0.000575   0.056698  
Ordinal('FXI3SAB_VOR_MM_A__y=2') 0         0.000364   0.340072  
                                 1         0.000247   0.092451  

[1342 rows x 3 columns]

## Processing Quantitative Features

For a binary target variable, following processing is applied:
* Pre-processing of quantitatives:
    - cutting feature into quantiles of sizes of at least `min_freq/2=5%` (train sample)
    - ordering modalities according to natural order
* All combinations of up to `max_n_mod=5` modalities are sorted by Tschuprow's T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/2=5%` frequency)
    - distinct target rate per consecutive modalities
    - no inversion of target rates between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [ ]:
from AutoCarver import Features, MulticlassCarver

features = Features(quantitatives=numerical_columns)

carver_numericals = MulticlassCarver(
    features=features,
    min_freq=0.10,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)
x_train = carver_numericals.fit_transform(
    x_train, y_transform(y_train), X_dev=x_dev, y_dev=y_transform(y_dev)
)

In [ ]:
# version = "010"
# carver_numericals.save(f"model/frequency/{version}_carver_numericals.json", light_mode=True)

In [10]:
from AutoCarver import MulticlassCarver

version = "010"
carver_numericals = MulticlassCarver.load(
    f"model/frequency/{version}_carver_numericals.json"
)
carver_numericals.summary

content  target_mean  \
feature                              label                              
Quantitative('ANCIENNETE__y=1')      0      x <= 0.0e+00     0.005282   
                                     1       0.0e+00 < x     0.006573   
Quantitative('TYPBAT2__y=1')         0      x <= 0.0e+00     0.004975   
                                     1       0.0e+00 < x     0.007069   
Quantitative('KAPITAL6__y=1')        0      x <= 0.0e+00     0.003562   
...                                                  ...          ...   
Quantitative('ALTITUDE_REGION__y=2') 1       9.7e-03 < x     0.000263   
Quantitative('Unknown__y=2')         0      x <= 1.0e+01     0.000265   
                                     1       1.0e+01 < x     0.000225   
Quantitative('Malveillance__y=2')    0      x <= 0.0e+00     0.000154   
                                     1       0.0e+00 < x     0.000260   

                                            frequency  
feature                              label             
Quantitative('ANCIENNETE__y=1')      0       0.128923  
                                     1       0.871077  
Quantitative('TYPBAT2__y=1')         0       0.316353  
                                     1       0.683647  
Quantitative('KAPITAL6__y=1')        0       0.573584  
...                                               ...  
Quantitative('ALTITUDE_REGION__y=2') 1       0.235004  
Quantitative('Unknown__y=2')         0       0.098212  
                                     1       0.855074  
Quantitative('Malveillance__y=2')    0       0.275733  
                                     1       0.677553  

[538 rows x 3 columns]

## Applying processing to samples

After this step:
* qualitative features can have up to 2 variants: one that maximizes association with target `y=1` and one with `y=2`
* quantitative features can hav up to 3 variants: one that maximizes association with target `y=1` and one with `y=2`, alongside their raw numerical values

In [11]:
x_train = carver_numericals.transform(x_train)
x_train = carver.transform(x_train)
x_dev = carver_numericals.transform(x_dev)
x_dev = carver.transform(x_dev)

Sanity check on OOS

In [12]:
import pandas as pd

data_path = "../data/"
oos = pd.read_csv(data_path + "test_input_5qJzHrr.csv")
oos = proc.transform(oos)
oos = carver.transform(oos)
oos = carver_numericals.transform(oos)

# # saving processed data
# version = "012"
# oos.to_csv(data_path+f'oos_frequency_{version}.csv')

<tmp>/ipykernel_9820\3208815277.py:4: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  oos = pd.read_csv(data_path+'test_input_5qJzHrr.csv')


# Feature Selection

## Selecting Quantitative Features

* Association with the target variable:
    - association is measured as Kruskal-Wallis' test statistic between the distributions of features for distinct target classes
    - features with associations lower than the `threshold=1` of Kruskal-Wallis' test statistic are removed
* Features are sorted according to their association with the target variable
* Association between features:
    - association is measured using Spearman's rho between quantitative features
    - features with associations greater than the `threshold=0.9` of Spearman's rho (in absolute value), with a feature more associated to the target variable, are removed
* The `n_best_per_type=50` best quantitative features are kept

In [14]:
from AutoCarver import Features
from AutoCarver.selectors import ClassificationSelector, KruskalMeasure, SpearmanFilter

# defining the features to select
quantitative_features = Features(quantitatives=numerical_columns)
print("number of quantitative features:", len(quantitative_features))

# defining the target association measures and threshold used to select features
measures = [KruskalMeasure(threshold=1)]

# defining the inter-feature association measures and threshold used to filter out features
filters = [SpearmanFilter(threshold=0.9)]

# initiating the selector
selector = ClassificationSelector(
    features=quantitative_features,
    measures=measures,
    filters=filters,
    n_best_per_type=50,
    max_num_features_per_chunk=250,
    verbose=True,
)

# feature selection on train data
best_quantitative_features = selector.select(x_train, y_transform(y_train))
print("number of selected quantitative features:", len(best_quantitative_features))

number of quantitative features: 217
 [ClassificationSelector] Selected Quantitative Features 


,feature,Nan,Mode,KruskalMeasure,KruskalRank,SpearmanFilter,SpearmanWith
46,Quantitative('KAPITAL32'),0.0000,0.3480,1027.4396,0.0000,0.0000,itself
26,Quantitative('KAPITAL12'),0.0000,0.3098,817.6330,1.0000,0.7784,KAPITAL32
58,Quantitative('SURFACE10'),0.0192,0.5240,792.0779,2.0000,0.6129,KAPITAL32
51,Quantitative('SURFACE1'),0.0000,0.0716,790.6300,3.0000,0.6405,KAPITAL32
55,Quantitative('SURFACE7'),0.0105,0.5593,739.8689,4.0000,0.6229,SURFACE10
35,Quantitative('KAPITAL21'),0.0141,0.5736,604.9863,5.0000,0.7058,KAPITAL32
73,Quantitative('NBBAT4'),0.0000,0.1214,593.8553,6.0000,0.8347,SURFACE1
86,Quantitative('NBSINSTRT'),0.0000,0.7274,504.3431,7.0000,0.3799,KAPITAL32
37,Quantitative('KAPITAL23'),0.0030,0.8343,436.8621,8.0000,0.4227,SURFACE7
96,Quantitative('EQUIPEMENT6'),0.0000,0.1006,369.5032,9.0000,0.3939,KAPITAL21


number of selected quantitative features: 50


## Selecting Qualitative Features

* Association with the target variable:
    - association is measured as Tschuprow's T with qualitative features
    - features with associations lower than the `threshold=0.005` of Tschuprow's T are removed
* Features are sorted according to their association with the target variable
* Association between features:
    - association is measured using Cramér's V between qualitative features
    - features with associations greater than the `threshold=0.9` of Cramér's V, with a feature more associated to the target variable, are removed
* The `n_best_per_type=50` best quantitative features are kept

In [ ]:
from AutoCarver import Features
from AutoCarver.selectors import (
    ClassificationSelector,
    CramervFilter,
    TschuprowtMeasure,
)


# defining the features to select
qualitative_features = Features(
    carver.features.versions + carver_numericals.features.versions
)
print("number of qualitative features:", len(qualitative_features))

# defining the target association measures and threshold used to select features
measures = [TschuprowtMeasure(threshold=0.005)]

# defining the inter-feature association measures and threshold used to filter out features
filters = [CramervFilter(threshold=0.9)]

# initiating the selector
selector = ClassificationSelector(
    features=qualitative_features,
    measures=measures,
    filters=filters,
    n_best_per_type=50,
    max_num_features_per_chunk=1000,
    verbose=True,
)

# feature selection on train data
best_qualitative_features = selector.select(x_train, y_train)
print("number of selected qualitative features:", len(best_qualitative_features))

number of qualitative features: 833
 [ClassificationSelector] Selected Qualitative Features 


,feature,Nan,Mode,TschuprowtMeasure,CramervFilter,CramervWith,TschuprowtRank
606,Categorical('KAPITAL32__y=1'),0.0000,0.5729,0.0365,0.0000,itself,0.0000
736,Categorical('SURFACE10__y=2'),0.0192,0.7855,0.0348,0.4739,KAPITAL32__y=1,1.0000
735,Categorical('SURFACE7__y=2'),0.0105,0.8035,0.0345,0.6119,SURFACE10__y=2,2.0000
730,Categorical('KAPITAL32__y=2'),0.0000,0.7945,0.0343,0.5890,KAPITAL32__y=1,3.0000
585,Categorical('SURFACE4__y=2'),0.0000,0.8253,0.0343,0.5637,KAPITAL32__y=2,4.0000
608,Categorical('SURFACE2__y=1'),0.0000,0.7114,0.0342,0.7224,SURFACE4__y=2,5.0000
609,Categorical('SURFACE3__y=1'),0.2289,0.5699,0.0339,0.8928,SURFACE2__y=1,6.0000
600,Categorical('KAPITAL12__y=1'),0.0000,0.5480,0.0337,0.5916,KAPITAL32__y=1,7.0000
613,Categorical('SURFACE11__y=1'),0.2289,0.6238,0.0335,0.8177,SURFACE3__y=1,8.0000
582,Categorical('TAILLE1__y=2'),0.0000,0.8700,0.0320,0.7232,SURFACE4__y=2,9.0000


number of selected qualitative features: 50


Saving selected features

In [ ]:
# import json

# version = "010"

# # listing selected features
# best_features = best_quantitative_features.versions + best_qualitative_features.versions

# # saving best features
# with open(f"model/frequency/{version}_best_features.json", "w", encoding="utf-8") as json_file:
#     json.dump(best_features, json_file)

In [17]:
import json

version = "010"

# laoding best features
with open(
    f"model/frequency/{version}_best_features.json", "r", encoding="utf-8"
) as json_file:
    best_features = json.load(json_file)

# XGBoost Modeling

## Hyperparameter Fine-Tuning with Bayesian Optimization

### Initial Approach: default XGBoost hyperparameters

In the following code, the XGBoost model's hyperparameters are fine-tuned using Optuna's implementation of Bayesian Optimization. This tuning process is designed to minimize the log loss on a development set in a multi-class classification context.

* A `xboost.XGBClassifier` is used with a `multi:softprob` objective function
* The weighted train sample is used to fit the model with the suggested hyperparameters
* The weighted dev sample is used to evaluate model performance
* The metric used for optimization is the multiclass logarithmic loss, computed with ``sklearn.metrics.log_loss``
* The default parameter ranges for optimization are defined as:

````python
XGB_PARAMS_RANGES = {
    "n_estimators": (100, 600),
    "learning_rate": (1e-4, 1),
    "max_depth": (1, 10),
    "min_child_weight": (1, 20),
    "subsample": (0.2, 1),
    "colsample_bytree": (0.2, 1),
    "colsample_bylevel": (0.2, 1),
    "gamma": (0, 10),
    "alpha": (0, 10),
    "lambda": (0, 10),
}
````

Key takeaways:
 * The Bayesian model is trained to predict output performance on dev sample from input hyperparameters
 * This implementation of the objective function ensures robustness of performances for selected hyperparameters

In [ ]:
import optuna

from objectives import get_multiclass_objective

# number of trials
N_TRIALS = 300

# defining the objective function of bayesian model
objective = get_multiclass_objective(
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    select_features=False,
)

# creating optuna study
study = optuna.create_study(direction="minimize")

# optimizing the objective function
study.optimize(objective, n_trials=N_TRIALS)

[I 2025-04-08 11:24:00,573] A new study created in memory with name: no-name-f4bc0bc8-03a8-4b90-9871-91720ce6b1d8
<home>/AppData\Local\pypoetry\Cache\virtualenvs\caa-challenge-frequency-BKzrcC_y-py3.11\Lib\site-packages\xgboost\core.py:158: UserWarning: [11:24:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
[I 2025-04-08 11:24:12,684] Trial 0 finished with value: 2.560983561036163 and parameters: {'objective': 'multi:softprob', 'n_estimators': 244, 'learning_rate':

In [83]:
study.best_params

{'objective': 'multi:softprob',
 'n_estimators': 101,
 'learning_rate': 0.0897475488338978,
 'max_depth': 2,
 'min_child_weight': 5,
 'subsample': 0.9841631212617082,
 'colsample_bytree': 0.35245975166168053,
 'colsample_bylevel': 0.5246351017520319,
 'gamma': 7.629531177690892,
 'alpha': 0.9464097613339071,
 'lambda': 0.8324174126752101}

In [ ]:
from objectives import get_best_multiclass_model

# Getting selected model
selected_features, model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE10', 'SURFACE1', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'NBSINCONJ', 'SURFACE17', 'KAPITAL8', 'TAILLE3', 'KAPITAL14', 'KAPITAL24', 'KAPITAL27', 'KAPITAL3', 'KAPITAL10', 'NBBAT2', 'EQUIPEMENT1', 'DEROG1', 'SURFACE13', 'CARACT5', 'NBBAT7', 'RISK5', 'NBBAT8', 'KAPITAL20', 'ZONE_VENT', 'KAPITAL17', 'KAPITAL25', 'RISK2', 'NBBAT6', 'ALTITUDE_2_num', 'ALTITUDE_5_num', 'DEROG11', 'RISK1', 'NBBAT9', 'KAPITAL1', 'DEROG10', 'DUREE_REQANEUF', 'DEROG7', 'EQUIPEMENT4', 'DEROG6', 'RISK4', 'IND_Y8_Y9_num', 'TYPBAT2', 'MEN_1IND_MEN_TOT', 'ALTITUDE_4_num', 'ALTITUDE_4_ALT_TOT', 'KAPITAL32__y=1', 'SURFACE10__y=2', 'SURFACE7__y=2', 'KAPITAL32__y=2', 'SURFACE4__y=2', 'SURFACE2__y=1', 'SURFACE3__y=1', 'KAPITAL12__y=1', 'SURFACE11__y=1', 'TAILLE1__y=2', 'SURFACE11__y=2', 'NBBAT4__y=1', 'KAPITAL6__y=2', 'TAILLE2__y=2', 'VOCATION__y=2', 'KAPITAL21__y=1', 'KAPITAL12__y=2', 'TYPERS__y=2', 'NBSINSTRT__y=2', 'NBBAT4__y=2', 'KAPI

### Advanced Approach: more Fine-Tuning

* Feature Selection:
  - Features are sorted according to their feature importance in the model
  - The number of features removed is added as a tunable hyperparameter (``n_features_removed``)
  - This enables automated feature pruning as part of the hyperparameter search process
* Hyperparameter ranges are constrained to close values to already selected ones: up to 10% away from selected values

Key takeaways:
* Removes unnecessary features that bring in noise to the model
* Helps the model to find the best hyperparameters in a narrower dimension space

In [97]:
from objectives import get_close_ranges, get_sorted_features

close_range = get_close_ranges(study.best_params)
sorted_features = get_sorted_features(model, best_features)

In [ ]:
from objectives import get_multiclass_objective

# number of trials
N_TRIALS = 300

# defining the objective function of bayesian model: with added feature selection and close range
objective = get_multiclass_objective(
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    sorted_features=sorted_features,
    custom_ranges=close_range,
)

# optimizing the objective function (keeping already existing study and trials)
study.optimize(objective, n_trials=N_TRIALS)

[I 2025-04-08 13:47:28,816] Trial 300 finished with value: 0.9230740457146489 and parameters: {'objective': 'multi:softprob', 'n_estimators': 99, 'learning_rate': 0.08219928492339859, 'max_depth': 2, 'min_child_weight': 5, 'subsample': 0.9906093570401614, 'colsample_bytree': 0.35333984496447923, 'colsample_bylevel': 0.5480637208924883, 'gamma': 6.927607000344657, 'alpha': 0.9916917170900429, 'lambda': 0.7777690901737532, 'n_features_removed': 15}. Best is trial 212 with value: 0.9193611106890472.
[I 2025-04-08 13:47:33,258] Trial 301 finished with value: 0.9314765782000503 and parameters: {'objective': 'multi:softprob', 'n_estimators': 94, 'learning_rate': 0.08194708208230411, 'max_depth': 2, 'min_child_weight': 5, 'subsample': 0.9793966805941682, 'colsample_bytree': 0.35402937233424825, 'colsample_bylevel': 0.5485111386792655, 'gamma': 6.936078155645755, 'alpha': 1.0268408312279158, 'lambda': 0.7687896620120332, 'n_features_removed': 16}. Best is trial 212 with value: 0.91936111068904

In [99]:
study.best_params

{'objective': 'multi:softprob',
 'n_estimators': 103,
 'learning_rate': 0.08940627199608095,
 'max_depth': 2,
 'min_child_weight': 5,
 'subsample': 0.9933943453305453,
 'colsample_bytree': 0.36557866317138143,
 'colsample_bylevel': 0.48540736226849523,
 'gamma': 8.25970225948475,
 'alpha': 0.9759013901022195,
 'lambda': 0.859040886684697,
 'n_features_removed': 0}

In [100]:
from objectives import get_best_multiclass_model

selected_features, model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE10', 'SURFACE1', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'NBSINCONJ', 'SURFACE17', 'KAPITAL8', 'TAILLE3', 'KAPITAL14', 'KAPITAL24', 'KAPITAL27', 'KAPITAL3', 'KAPITAL10', 'NBBAT2', 'EQUIPEMENT1', 'DEROG1', 'SURFACE13', 'CARACT5', 'NBBAT7', 'RISK5', 'NBBAT8', 'KAPITAL20', 'ZONE_VENT', 'KAPITAL17', 'KAPITAL25', 'RISK2', 'NBBAT6', 'ALTITUDE_2_num', 'ALTITUDE_5_num', 'DEROG11', 'RISK1', 'NBBAT9', 'KAPITAL1', 'DEROG10', 'DUREE_REQANEUF', 'DEROG7', 'EQUIPEMENT4', 'DEROG6', 'RISK4', 'IND_Y8_Y9_num', 'TYPBAT2', 'MEN_1IND_MEN_TOT', 'ALTITUDE_4_num', 'ALTITUDE_4_ALT_TOT', 'KAPITAL32__y=1', 'SURFACE10__y=2', 'SURFACE7__y=2', 'KAPITAL32__y=2', 'SURFACE4__y=2', 'SURFACE2__y=1', 'SURFACE3__y=1', 'KAPITAL12__y=1', 'SURFACE11__y=1', 'TAILLE1__y=2', 'SURFACE11__y=2', 'NBBAT4__y=1', 'KAPITAL6__y=2', 'TAILLE2__y=2', 'VOCATION__y=2', 'KAPITAL21__y=1', 'KAPITAL12__y=2', 'TYPERS__y=2', 'NBSINSTRT__y=2', 'NBBAT4__y=2', 'KAPI

<home>/AppData\Local\pypoetry\Cache\virtualenvs\caa-challenge-frequency-BKzrcC_y-py3.11\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:26:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_features_removed" } are not used.

  warnings.warn(smsg, UserWarning)


Log Loss Train: 0.7869919838903734
Log Loss Dev:   0.9133600293223422


### LEGACY: OLDER MODELS

In [19]:
from objectives import get_best_multiclass_model

selected_features, model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'KAPITAL14', 'KAPITAL8', 'TAILLE3', 'SURFACE17', 'KAPITAL27', 'KAPITAL10', 'KAPITAL3', 'NBBAT10', 'ZONE_VENT', 'KAPITAL25', 'ALTITUDE_5_num', 'RISK1', 'NBBAT6', 'VOCATION__y=1', 'DEROG12__y=1', 'KAPITAL43__y=1', 'DEROG4__y=1', 'RISK10__y=1', 'ACTIVIT2__y=1', 'total_surface_crossed__y=1', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=1', 'NBJTX0_MM_A_NBJTX0_MMAX_A__y=1', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A__y=1', 'NBJRR50_MM_A_NBJRR50_MMAX_A__y=1', 'EQUIPEMENT2__y=1', 'TAILLE1__y=1', 'TAILLE1__y=2', 'SURFACE6__y=2', 'HAUTEUR_MAX__y=2', 'BDTOPO_BAT_MAX_HAUTEUR__y=2', 'HAUTEUR__y=2']
Log Loss Train: 0.8794817435042118
Log Loss Dev:   0.9277229259014648
Log Loss Overall:   0.8771232504614386


In [30]:
from objectives import get_best_multiclass_model

model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train <= 1, 2),
    x_dev[best_features],
    y_dev.where(y_dev <= 1, 2),
    w_train=w_train,
    w_dev=w_dev,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE1', 'SURFACE10', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'NBSINCONJ', 'KAPITAL14', 'KAPITAL8', 'TAILLE3', 'SURFACE17', 'KAPITAL27', 'KAPITAL24', 'KAPITAL10', 'KAPITAL3', 'EQUIPEMENT1', 'NBBAT10', 'DEROG1', 'NBBAT8', 'ZONE_VENT', 'CARACT5', 'SURFACE13', 'DEROG11', 'RISK5', 'KAPITAL20', 'DEROG10', 'KAPITAL25', 'DUREE_REQANEUF', 'ALTITUDE_5_num', 'NBBAT9', 'RISK2', 'RISK1', 'ALTITUDE_3_num', 'KAPITAL17', 'NBBAT6', 'KAPITAL40__y=2', 'VOCATION__y=1', 'TYPERS__y=2', 'KAPITAL41__y=2', 'DEROG12__y=1', 'KAPITAL42__y=2', 'RISK11__y=1', 'KAPITAL43__y=1', 'KAPITAL37__y=2', 'DEROG4__y=1', 'RISK10__y=1', 'ACTIVIT2__y=1', 'TYPBAT1__y=2', 'DEROG5__y=1', 'total_surface_crossed__y=1', 'AN_EXERC__y=1', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=1', 'TXAB_VOR_MM_A_TXAB_VOR_MMAX_A__y=1', 'NBJTX0_MM_A_NBJTX0_MMAX_A__y=1', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A__y=1', 'NBJRR50_MM_A_NBJRR50_MMAX_A__y=1', 'EQUIPEMENT2__y=1', 'NBJFF10_

In [61]:
from objectives import get_best_multiclass_model

model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train <= 1, 2),
    x_dev[best_features],
    y_dev.where(y_dev <= 1, 2),
    w_train=w_train,
    w_dev=w_dev,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE1', 'SURFACE10', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'NBSINCONJ', 'KAPITAL14', 'KAPITAL8', 'TAILLE3', 'SURFACE17', 'KAPITAL27', 'KAPITAL24', 'KAPITAL10', 'KAPITAL3', 'EQUIPEMENT1', 'NBBAT10', 'DEROG1', 'NBBAT8', 'ZONE_VENT', 'CARACT5', 'SURFACE13', 'DEROG11', 'RISK5', 'KAPITAL20', 'DEROG10', 'KAPITAL25', 'DUREE_REQANEUF', 'ALTITUDE_5_num', 'NBBAT9', 'RISK2', 'RISK1', 'ALTITUDE_3_num', 'KAPITAL17', 'NBBAT6', 'KAPITAL40__y=2', 'VOCATION__y=1', 'TYPERS__y=2', 'KAPITAL41__y=2', 'KAPITAL42__y=2', 'RISK11__y=1', 'KAPITAL43__y=1', 'KAPITAL37__y=2', 'NBJFF10_MM_A_NBJFF10_MMAX_A__y=1', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=2', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=1', 'NBJFF28_MM_A_NBJFF28_MMAX_A__y=1', 'NBJTNI10_MM_A_NBJTNI10_MMAX_A__y=1', 'DEROG4__y=1', 'NBJFXY10_MM_A_NBJFXY10_MMAX_A__y=1', 'TXMIN_VOR_MM_A_TXMIN_VOR_MMAX_A__y=1', 'RRAB_VOR_MM_A_RRAB_VOR_MMAX_A__y=2', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A__y

In [ ]:
from objectives import get_best_multiclass_model, get_best_poisson_model

model = get_best_poisson_model(
    study.best_params,
    x_train[best_features],
    y_train,
    x_dev[best_features],
    y_dev,
    w_train=w_train,
    w_dev=w_dev,
)

used features: ['KAPITAL32', 'KAPITAL12', 'SURFACE1', 'SURFACE10', 'SURFACE7', 'KAPITAL21', 'NBBAT4', 'NBSINSTRT', 'KAPITAL23', 'EQUIPEMENT6', 'NBSINCONJ', 'KAPITAL14', 'KAPITAL8', 'TAILLE3', 'SURFACE17', 'KAPITAL27', 'KAPITAL24', 'KAPITAL10', 'KAPITAL3', 'EQUIPEMENT1', 'NBBAT10', 'DEROG1', 'NBBAT8', 'ZONE_VENT', 'CARACT5', 'SURFACE13', 'DEROG11', 'RISK5', 'KAPITAL20', 'DEROG10', 'KAPITAL25', 'DUREE_REQANEUF', 'ALTITUDE_5_num', 'NBBAT9', 'RISK2', 'RISK1', 'ALTITUDE_3_num', 'KAPITAL17', 'NBBAT6', 'KAPITAL40__y=2', 'VOCATION__y=1', 'TYPERS__y=2', 'KAPITAL41__y=2', 'KAPITAL42__y=2', 'RISK11__y=1', 'KAPITAL43__y=1', 'KAPITAL37__y=2', 'NBJFF10_MM_A_NBJFF10_MMAX_A__y=1', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=2', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A__y=1', 'NBJFF28_MM_A_NBJFF28_MMAX_A__y=1', 'NBJTNI10_MM_A_NBJTNI10_MMAX_A__y=1', 'DEROG4__y=1', 'NBJFXY10_MM_A_NBJFXY10_MMAX_A__y=1', 'TXMIN_VOR_MM_A_TXMIN_VOR_MMAX_A__y=1', 'RRAB_VOR_MM_A_RRAB_VOR_MMAX_A__y=2', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A__y

### Saving fine-tuned model

In [ ]:
# version = "012"

# model.save_model(f"model/frequency/{version}_xgboost.json")

# with open(f"model/frequency/{version}_selected_features.json", "w", encoding="utf-8") as json_file:
#     json.dump(selected_features, json_file)

# Prediction on OOS

* The trained model predicts the probability of observations to be in each of the three classes
* To get the actual frequency the law of total expectation is used:
$$
\mathbb{E}[Y] = \mathbb{P}(X = 0) \cdot \mathbb{E}[Y \mid X = 0] + \mathbb{P}(X = 1) \cdot \mathbb{E}[Y \mid X = 1] + \mathbb{P}(X = 2) \cdot \mathbb{E}[Y \mid X = 2]
$$

With the following values computed on the whole train and dev datasets:

$$
\mathbb{E}[Y \mid X = 0] = 0
$$
$$
\mathbb{E}[Y \mid X = 1] = 1
$$
$$
\mathbb{E}[Y \mid X = 2] = 2.076923
$$

In [ ]:
from xgboost import XGBClassifier


# loading trained model
version = "012"
model = XGBClassifier()
model.load_model(f"model/frequency/{version}_xgboost.json")

# prediction on OOS
oos_pred = model.predict_proba(oos[best_features])
oos_pred

array([[0.27556354, 0.38055715, 0.34387934],
       [0.4185339 , 0.366023  , 0.21544312],
       [0.61103684, 0.31478277, 0.07418045],
       ...,
       [0.45064056, 0.34262392, 0.20673557],
       [0.42191902, 0.45581764, 0.12226339],
       [0.69764143, 0.20738934, 0.09496926]], dtype=float32)

In [26]:
# computing the mean of the target variable for each class
data.groupby(y_transform(data[target_col]))[target_col].mean()

TARGET
0    0.000000
1    1.000000
2    2.076923
Name: TARGET, dtype: float64

Processed OOS sample and predictions are saved for uses within the amounts model

In [ ]:
# adding model predictions
oos["pred_1"] = oos_pred[:, 1]
oos["pred_2"] = oos_pred[:, 2]
oos["pred_sum"] = (oos_pred * np.array([0, 1, 2.076923076923077])).sum(axis=1)

# saving processed OOS sample
version = "012"
oos.to_csv(data_path + f"oos_frequency_{version}.csv", index=False)

Processed train/dev samples and predictions are saved for uses within the amounts model

In [ ]:
# prediction on train and dev
x_train["pred_1"] = model.predict_proba(x_train[best_features])[:, 1]
x_train["pred_2"] = model.predict_proba(x_train[best_features])[:, 2]
x_train["pred_sum"] = (
    model.predict_proba(x_train[best_features]) * np.array([0, 1, 2.076923076923077])
).sum(axis=1)
x_dev["pred_1"] = model.predict_proba(x_dev[best_features])[:, 1]
x_dev["pred_2"] = model.predict_proba(x_dev[best_features])[:, 2]
x_dev["pred_sum"] = (
    model.predict_proba(x_dev[best_features]) * np.array([0, 1, 2.076923076923077])
).sum(axis=1)

# merging back train and dev
merged = pd.concat([x_train, x_dev], axis=0)

# saving dataset
version = "012"
merged.to_csv(data_path + f"frequency_{version}.csv")

## Creating Output for Challenge Submission

In [28]:
# converting multiclass to frequency
oos_pred = (oos_pred * np.array([0, 1, 2.076923076923077])).sum(axis=1)
oos_pred

array([1.09476809, 0.81348179, 0.46884985, ..., 0.77199779, 0.7097493 ,
       0.40463318])

In [ ]:
import pandas as pd

# filtering out unnecessary columns
pred = oos[["ID", "ANNEE_ASSURANCE"]]

# adding in freq predictions
pred["FREQ"] = oos_pred

# saving predictions
version = "025"
pred.to_csv(f"predictions/temp/{version}_pred.csv", index=False)

<tmp>/ipykernel_26904\133441544.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred["FREQ"] = oos_pred


# Target Processing for the Claim Amount Model

Claim amount is discretized to maximize the association with the following target:
* `0`: exactly one claim
*  `1`: two or more claims

Only observations with claims are used for this purpose.

This will be used to stratify sampling for the claim amount model.  

In [7]:
(y_transform(y_train)[y_train > 0] - 1).value_counts(normalize=False).sort_index()

TARGET
0    1966
1      73
Name: count, dtype: int64

In [5]:
(y_transform(y_train)[y_train > 0] - 1).value_counts(normalize=True).sort_index()

TARGET
0    0.964198
1    0.035802
Name: proportion, dtype: float64

In [6]:
from AutoCarver import BinaryCarver, Features

# selecting target of amount model to discretize
features = Features(quantitatives=["CM"])

# initiating the carver
target_carver = BinaryCarver(
    features=features,
    min_freq=0.05,
    max_n_mod=5,
    dropna=False,
    copy=True,
    verbose=True,
)

# discretizing the target variable
target_carver.fit_transform(
    x_train[y_train > 0],
    (y_transform(y_train)[y_train > 0] - 1),
    X_dev=x_dev[y_dev > 0],
    y_dev=(y_transform(y_dev)[y_dev > 0] - 1),
)

------
--- [QuantitativeDiscretizer] Fit Features(['CM'])
 - [ContinuousDiscretizer] Fit Features(['CM'])
 - [OrdinalDiscretizer] Fit Features(['CM'])
------

---------
------ [BinaryCarver] Fit Features(['CM'])
--- [BinaryCarver] Fit Quantitative('CM') (1/1)
 [BinaryCarver] Raw distribution


,target_mean,frequency
x <= 0.00e+00,0.0259,0.1893
0.00e+00 < x <= 1.00e+02,0.0169,0.0289
1.00e+02 < x <= 1.51e+02,0.0682,0.0216
1.51e+02 < x <= 2.32e+02,0.0192,0.0255
2.32e+02 < x <= 3.04e+02,0.0784,0.0250
3.04e+02 < x <= 4.00e+02,0.0123,0.0799
4.00e+02 < x <= 4.97e+02,0.0385,0.0255
4.97e+02 < x <= 5.92e+02,0.0196,0.0250
5.92e+02 < x <= 6.01e+02,0.0000,0.0250
6.01e+02 < x <= 7.29e+02,0.0385,0.0255


Testing robustness    :   0%|          | 132/31930 [00:00<01:01, 515.02it/s]




 [BinaryCarver] Carved distribution


X distribution 
 
 
   
 target_mean 
 frequency 
 
 
 
 
 x <= 6.01e+02 
 0.0264 
 0.4458 
 
 
 6.01e+02 < x <= 8.69e+02 
 0.0680 
 0.0505 
 
 
 8.69e+02 < x <= 1.49e+03 
 0.0260 
 0.0755 
 
 
 1.49e+03 < x <= 5.14e+04 
 0.0495 
 0.3271 
 
 
 5.14e+04 < x 
 0.0243 
 0.1010 
 
 
 
 
 
 X_dev distribution 
 
 
 target_mean 
 frequency 
 
 
 
 
 0.0359 
 0.4373 
 
 
 0.0909 
 0.0647 
 
 
 0.0244 
 0.0804 
 
 
 0.0382 
 0.3078 
 
 
 0.0000 
 0.1098

,ACTIVIT2,VOCATION,TYPERS,ANCIENNETE,ADOSS,CARACT1,CARACT2,CARACT3,INDEM1,DUREE_REQANEUF,...,RRAB_VOR_MM_A,RRAB_VOR_MMAX_A,ANNEE_ASSURANCE,ESPINSEE,AN_EXERC,ZONE,FREQ,CM,CHARGE,TARGET
ID,,,,,,,,,,,,,,,,,,,,,
191738,ACT1,VOC6,2,9,N,N,NaN,NaN,N,2.0,...,NaN,NaN,1.000000,ESP3,ANNEE1,39,1.000000,0.0,0.00,1
343260,ACT1,VOC6,2,11,N,N,NaN,NaN,N,2.0,...,NaN,NaN,1.000000,ESP3,ANNEE6,15,1.000000,0.0,417.00,1
188593,ACT1,VOC6,1,3,N,N,NaN,NaN,N,2.0,...,NaN,NaN,1.000000,ESP3,ANNEE1,58,1.000000,0.0,355.49,1
134860,ACT1,VOC6,1,2,N,N,NaN,NaN,N,2.0,...,03. <= 25,02. <= 57,1.512329,NaN,ANNEE1,93,1.322464,0.0,0.00,2
4047,ACT5,VOC6,2,8,N,R,NaN,NaN,N,NaN,...,NaN,NaN,1.000000,NaN,ANNEE5,30,1.000000,0.0,143.86,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108262,ACT1,VOC6,1,2,N,N,NaN,NaN,N,2.0,...,03. <= 25,02. <= 57,1.000000,NaN,ANNEE2,32,1.000000,3.0,6750.00,1
62835,ACT1,VOC6,2,8,N,R,NaN,NaN,N,2.0,...,02. <= 19,01. <= 41,0.720548,NaN,ANNEE3,89,1.387833,0.0,178.90,1
261058,ACT1,VOC6,2,11,N,R,NaN,NaN,N,2.0,...,03. <= 25,04. >= 82,1.000000,ESP3,ANNEE8,15,1.000000,2.0,1332.00,1


In [ ]:
from AutoCarver import BinaryCarver, Features

# selecting target of amount model to discretize
features = Features(quantitatives=["CM"])

# initiating the carver
target_carver = BinaryCarver(
    features=features,
    min_freq=0.05,
    max_n_mod=5,
    dropna=False,
    copy=True,
    verbose=True,
)

# discretizing the target variable
target_carver.fit_transform(
    x_train[y_train > 0],
    (y_transform(y_train)[y_train > 0] - 1),
    X_dev=x_dev[y_dev > 0],
    y_dev=(y_transform(y_dev)[y_dev > 0] - 1),
)

------
--- [QuantitativeDiscretizer] Fit Features(['CM'])
 - [ContinuousDiscretizer] Fit Features(['CM'])
 - [OrdinalDiscretizer] Fit Features(['CM'])
------

---------
------ [BinaryCarver] Fit Features(['CM'])
--- [BinaryCarver] Fit Quantitative('CM') (1/1)
 [BinaryCarver] Raw distribution


,target_mean,frequency
x <= 0.00e+00,0.0259,0.1893
0.00e+00 < x <= 1.00e+02,0.0169,0.0289
1.00e+02 < x <= 1.51e+02,0.0682,0.0216
1.51e+02 < x <= 2.32e+02,0.0192,0.0255
2.32e+02 < x <= 3.04e+02,0.0784,0.0250
3.04e+02 < x <= 4.00e+02,0.0123,0.0799
4.00e+02 < x <= 4.97e+02,0.0385,0.0255
4.97e+02 < x <= 5.92e+02,0.0196,0.0250
5.92e+02 < x <= 6.01e+02,0.0000,0.0250
6.01e+02 < x <= 7.29e+02,0.0385,0.0255


Testing robustness    :   0%|          | 132/31930 [00:00<01:02, 508.15it/s]



 [BinaryCarver] Carved distribution


X distribution 
 
 
   
 target_mean 
 frequency 
 
 
 
 
 x <= 6.01e+02 
 0.0264 
 0.4458 
 
 
 6.01e+02 < x <= 8.69e+02 
 0.0680 
 0.0505 
 
 
 8.69e+02 < x <= 1.49e+03 
 0.0260 
 0.0755 
 
 
 1.49e+03 < x <= 5.14e+04 
 0.0495 
 0.3271 
 
 
 5.14e+04 < x 
 0.0243 
 0.1010 
 
 
 
 
 
 X_dev distribution 
 
 
 target_mean 
 frequency 
 
 
 
 
 0.0359 
 0.4373 
 
 
 0.0909 
 0.0647 
 
 
 0.0244 
 0.0804 
 
 
 0.0382 
 0.3078 
 
 
 0.0000 
 0.1098

,ACTIVIT2,VOCATION,TYPERS,ANCIENNETE,ADOSS,CARACT1,CARACT2,CARACT3,INDEM1,DUREE_REQANEUF,...,zone_vent_extinction_rate,Unknown,Involontaire (travaux),Involontaire (particulier),Accidentelle,Malveillance,Naturelle,VENT_x_CASERNES,KAPITAL_SUM,KAPITAL_MAX
ID,,,,,,,,,,,,,,,,,,,,,
191738,ACT1,VOC6,2,9,N,N,NaN,NaN,N,2.0,...,1.0__Aucun feu,76.0,11.0,4.0,8.0,3.0,0.0,1.0__01. <= 1,94001.0,22500.0
343260,ACT1,VOC6,2,11,N,N,NaN,NaN,N,2.0,...,3.0__<50%,184.0,54.0,34.0,7.0,7.0,2.0,3.0__01. <= 1,401501.0,125000.0
188593,ACT1,VOC6,1,3,N,N,NaN,NaN,N,2.0,...,2.0__50-70%,26.0,3.0,7.0,2.0,6.0,0.0,2.0__01. <= 1,568003.0,125000.0
134860,ACT1,VOC6,1,2,N,N,NaN,NaN,N,2.0,...,2.0__Aucun feu,NaN,NaN,NaN,NaN,NaN,NaN,2.0__01. <= 1,681001.0,175000.0
4047,ACT5,VOC6,2,8,N,R,NaN,NaN,N,NaN,...,1.0__>85%,388.0,85.0,48.0,45.0,126.0,19.0,1.0__01. <= 1,249503.0,175000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108262,ACT1,VOC6,1,2,N,N,NaN,NaN,N,2.0,...,3.0__<50%,37.0,2.0,2.0,1.0,1.0,0.0,3.0__01. <= 1,533001.0,350000.0
62835,ACT1,VOC6,2,8,N,R,NaN,NaN,N,2.0,...,2.0__70-85%,49.0,4.0,1.0,5.0,3.0,0.0,2.0__01. <= 1,493002.0,300000.0
261058,ACT1,VOC6,2,11,N,R,NaN,NaN,N,2.0,...,3.0__<50%,184.0,54.0,34.0,7.0,7.0,2.0,3.0__01. <= 1,199501.0,175000.0


In [ ]:
target_carver.summary

content  \
feature            cramerv  tschuprowt n_mod label                             
Quantitative('CM') 0.070898 0.050132   5     0                 x <= 6.01e+02   
                                             1      6.01e+02 < x <= 8.69e+02   
                                             2      8.69e+02 < x <= 1.49e+03   
                                             3      1.49e+03 < x <= 5.14e+04   
                                             4                  5.14e+04 < x   

                                                    target_mean  frequency  
feature            cramerv  tschuprowt n_mod label                          
Quantitative('CM') 0.070898 0.050132   5     0         0.026403   0.445807  
                                             1         0.067961   0.050515  
                                             2         0.025974   0.075527  
                                             3         0.049475   0.327121  
                                             4         0.024272   0.101030

In [ ]:
# saving carver
target_carver.save("model/cm_carver_freq_tschuprowt.json")